# Modelo XGBoost con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
XGBoost con MultiOutputRegressor y búsqueda de hiperparámetros.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "xgboost")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1

XGB_PARAM_GRID = {
    'estimator__n_estimators': [100, 200, 300],
    'estimator__max_depth': [3, 6, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0],
    'estimator__reg_alpha': [0, 0.1, 1],
    'estimator__reg_lambda': [1, 1.5, 2]
}


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def get_station_from_features(X_sample, feature_names, station_prefix='Estacion_'):
    n_features = len(feature_names)
    first_block = X_sample[:n_features]
    station_indices = [i for i, name in enumerate(feature_names) if name.startswith(station_prefix)]
    for idx in station_indices:
        if abs(first_block[idx] - 1.0) < 0.1:
            return feature_names[idx][len(station_prefix):]
    return None


def compute_per_station_metrics(y_true, y_pred, X_test, feature_names, output_dir, entity_name):
    n_samples = len(y_true)
    station_preds = {}
    for i in range(n_samples):
        station = get_station_from_features(X_test[i], feature_names)
        if station is None:
            continue
        station_preds.setdefault(station, {'true': [], 'pred': []})
        station_preds[station]['true'].append(y_true[i])
        station_preds[station]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                           entity_name, output_subdir, feature_names=None, original_csv_path=None):
    print(f"\n--- Entrenando XGBoost para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: sin datos de validación.")
        return None, None

    base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE)
    multi_model = MultiOutputRegressor(base_model, n_jobs=1)
    tscv = TimeSeriesSplit(n_splits=3)
    grid_search = GridSearchCV(estimator=multi_model, param_grid=XGB_PARAM_GRID, cv=tscv,
                               scoring='neg_mean_squared_error', n_jobs=N_JOBS, verbose=2)
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    best_params = {k.replace('estimator__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"  Mejores parámetros: {best_params}")

    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())
    print(f"  Métricas en validación: R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if feature_names is not None and original_csv_path is not None:
        station_metrics = compute_per_station_metrics(y_test, test_pred, X_test, feature_names, output_subdir, entity_name)
    else:
        station_metrics = None

    with open(os.path.join(output_subdir, "model.pkl"), 'wb') as f:
        pickle.dump(best_model, f)
    results = {'best_params': best_params, 'validation_metrics': val_metrics, 'test_metrics': test_metrics,
               'station_metrics': station_metrics, 'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test, test_pred, horizons, os.path.join(output_subdir, "test_scatter.png"),
                     f"XGBoost - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)
    return best_model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST POR TRANSECTO")
    ml_2d_dir = os.path.join(DATA_DIR, "by_transect", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        original_csv = os.path.join(ENCODED_DIR, "ml", "by_transect", f"{entity}.csv")
        feature_names = None
        if os.path.exists(original_csv):
            df_cols = pd.read_csv(original_csv, nrows=0, index_col=0)
            feature_names = df_cols.columns.tolist()

        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names, original_csv if os.path.exists(original_csv) else None)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST GLOBAL")
    ml_2d_dir = os.path.join(DATA_DIR, "global", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names=None, original_csv_path=None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("XGBOOST")
    process_by_transect()
    process_global()
    generate_summary()